In [2]:
# testing CNN-LSTM
# extracet seklton
# using ball detection as trigger

!pip install -q mediapipe==0.10.14 opencv-python-headless

import os
import cv2
import numpy as np
import mediapipe as mp
from collections import Counter
from tensorflow.keras.models import load_model
from google.colab import files

print("All packages")

All packages


In [3]:
print("Step 1: Upload tennis_cnn_lstm_v1.h5")
uploaded = files.upload()
MODEL_FILE = [f for f in uploaded.keys() if f.endswith('.h5')][0]
model = load_model(MODEL_FILE)
print(f" Model loaded: {MODEL_FILE}")
model.summary()


print("\nStep 2: Upload test video (e.g. FF7.mp4)")
uploaded = files.upload()
VIDEO_FILE = [f for f in uploaded.keys() if f.endswith('.mp4')][0]
print(f" Video loaded: {VIDEO_FILE}")

Step 1: Upload tennis_cnn_lstm_v1.h5


Saving tennis_cnn_lstm_v1.h5 to tennis_cnn_lstm_v1.h5


 Model loaded: tennis_cnn_lstm_v1.h5


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 30, 64)         │        22,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

 Trainable params: 35,843 (140.01 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)


Step 2: Upload test video (e.g. FF7.mp4)


Saving FF7.mp4 to FF7.mp4
 Video loaded: FF7.mp4


In [4]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.6
)



SELECTED_JOINTS = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27]


cap = cv2.VideoCapture(VIDEO_FILE)
all_skeletons = []
frame_count = 0

print(f"Extracting skeleton from {VIDEO_FILE}...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        pts = [[lm[i].x, lm[i].y] for i in SELECTED_JOINTS]
    else:

        pts = all_skeletons[-1].tolist() if all_skeletons else [[0.0, 0.0]] * len(SELECTED_JOINTS)

    all_skeletons.append(pts)
    frame_count += 1

cap.release()
pose.close()

skeleton = np.array(all_skeletons)
np.save("match_skeleton.npy", skeleton)

print(f"Skeleton extracted → shape: {skeleton.shape}")
print(f"   ({frame_count} frames processed)")

Extracting skeleton from FF7.mp4...


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Skeleton extracted → shape: (618, 11, 2)
   (618 frames processed)


In [6]:
#  HSV ball detection
def detect_ball_hsv(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower_yellow = np.array([25, 120, 140])
    upper_yellow = np.array([40, 255, 255])
    mask = cv2.inRange(hsv, lower_yellow, upper_yellow)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best = None
    best_score = 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 40 or area > 600:
            continue
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue
        circularity = 4 * np.pi * area / (perimeter ** 2)
        score = area * circularity
        if score > best_score and circularity > 0.65:
            best_score = score
            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                best = (cx, cy)
    return best



def is_ball_near_player(ball_pos, skel_frame, width, height):

    if ball_pos is None:
        return False
    bx = ball_pos[0] / width
    by = ball_pos[1] / height
    for idx in [0, 1, 2, 3, 4, 5]:
        jx, jy = skel_frame[idx]
        dist = np.sqrt((bx - jx)**2 + (by - jy)**2)
        if dist < 0.25:
            return True
    return False


print("Helper functions defined")


Helper functions defined


In [10]:

THRESHOLD       = 0.82
COOLDOWN_FRAMES = 60
WINDOW_SIZE     = 30
STRIDE          = 5

LABEL_MAP    = {0: "Forehand", 1: "Backhand", 2: "Serve"}
LABEL_COLORS = {0: (0, 200, 80), 1: (200, 80, 0), 2: (0, 80, 255)}  # BGR


skeleton = np.load("match_skeleton.npy")
print("Skeleton shape:", skeleton.shape)


cap = cv2.VideoCapture(VIDEO_FILE)
fps    = cap.get(cv2.CAP_PROP_FPS) or 30
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

frame_list = []
while cap.isOpened():
    ret, fr = cap.read()
    if not ret:
        break
    frame_list.append(fr)
cap.release()
print(f"Loaded {len(frame_list)} frames  |  {fps:.1f} fps  |  {width}×{height}")


print("Detecting ball positions...")
ball_positions = [detect_ball_hsv(fr) for fr in frame_list]
ball_found = sum(1 for b in ball_positions if b is not None)
print(f"Ball detected in {ball_found}/{len(frame_list)} frames")


output_name = VIDEO_FILE.replace(".mp4", "") + "_detected.mp4"
out = cv2.VideoWriter(output_name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))


detections = []
last_detection_frame = -COOLDOWN_FRAMES
buffer = []

print("Running stroke detection...")

for i in range(len(skeleton)):
    pts = skeleton[i]
    buffer.append(pts.flatten())

    if len(buffer) > WINDOW_SIZE:
        buffer.pop(0)


    if len(buffer) == WINDOW_SIZE and (i % STRIDE == 0) and (i - last_detection_frame) >= COOLDOWN_FRAMES:
        seq = np.array(buffer)


        hip_x = seq[:, 12]
        hip_y = seq[:, 13]
        seq_3d  = seq.reshape(WINDOW_SIZE, 11, 2)
        hip_3d  = np.stack([hip_x, hip_y], axis=1)[:, None, :]
        seq_3d  = seq_3d - hip_3d
        seq_input = seq_3d.reshape(1, WINDOW_SIZE, 22)


        probs = model.predict(seq_input, verbose=0)[0]
        cls   = int(np.argmax(probs))
        conf  = float(probs[cls])

        if conf >= THRESHOLD:

            ball_near = any(
                is_ball_near_player(ball_positions[j], skeleton[j], width, height)
                for j in range(max(0, i - WINDOW_SIZE), i + 1)
            )


            if ball_near:
                detections.append((i, cls, conf))
                last_detection_frame = i
                print(f"  Frame {i:5d} → {LABEL_MAP[cls]:8s}  conf={conf:.3f}")


    canvas = frame_list[i].copy()


    for j in range(len(pts)):
        x = int(pts[j][0] * width)
        y = int(pts[j][1] * height)
        cv2.circle(canvas, (x, y), 5, (255, 255, 255), -1)


    if ball_positions[i]:
        bx, by = ball_positions[i]
        cv2.circle(canvas, (bx, by), 12, (0, 255, 255), 3)


    if detections and detections[-1][0] <= i < detections[-1][0] + WINDOW_SIZE:
        label = LABEL_MAP[detections[-1][1]]
        color = LABEL_COLORS[detections[-1][1]]

        cv2.putText(canvas, label.upper(), (80, 140),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.8, (0, 0, 0), 10)
        cv2.putText(canvas, label.upper(), (80, 140),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.8, color, 6)


    count_text = f"F:{sum(1 for d in detections if d[1]==0)}  B:{sum(1 for d in detections if d[1]==1)}  S:{sum(1 for d in detections if d[1]==2)}"
    cv2.putText(canvas, count_text, (width - 320, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)
    cv2.putText(canvas, count_text, (width - 320, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 1)

    out.write(canvas)


f_count = sum(1 for d in detections if d[1] == 0)
b_count = sum(1 for d in detections if d[1] == 1)
s_count = sum(1 for d in detections if d[1] == 2)
total   = len(detections)

stats_frame = np.zeros((height, width, 3), dtype=np.uint8)
lines = [
    ("STROKE SUMMARY", (255, 255, 255), 1.8, 80),
    (f"Forehand  : {f_count}", (0, 200, 80),  1.5, 160),
    (f"Backhand  : {b_count}", (200, 80, 0),  1.5, 230),
    (f"Serve     : {s_count}", (0, 80, 255),  1.5, 300),
    (f"TOTAL     : {total}",   (255, 255, 0), 1.6, 390),
]
for text, color, scale, y_pos in lines:
    cv2.putText(stats_frame, text, (width // 2 - 250, y_pos),
                cv2.FONT_HERSHEY_SIMPLEX, scale, color, 3)

for _ in range(int(fps * 3)):
    out.write(stats_frame)

out.release()


print("\n" + "=" * 60)
print("DETECTION COMPLETE")
print(f"  Forehand : {f_count}")
print(f"  Backhand : {b_count}")
print(f"  Serve    : {s_count}")
print(f"  TOTAL    : {total}")
print(f"Output → {output_name}")
print("=" * 60)


files.download(output_name)

Skeleton shape: (618, 11, 2)
Loaded 618 frames  |  62.0 fps  |  848×392
Detecting ball positions...
Ball detected in 391/618 frames
Running stroke detection...
  Frame    30 → Forehand  conf=1.000
  Frame    90 → Forehand  conf=1.000
  Frame   150 → Backhand  conf=1.000
  Frame   210 → Forehand  conf=1.000
  Frame   270 → Forehand  conf=1.000
  Frame   330 → Forehand  conf=1.000
  Frame   390 → Forehand  conf=1.000
  Frame   450 → Forehand  conf=1.000
  Frame   510 → Forehand  conf=1.000
  Frame   570 → Forehand  conf=1.000

DETECTION COMPLETE
  Forehand : 9
  Backhand : 1
  Serve    : 0
  TOTAL    : 10
Output → FF7_detected.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>